In [21]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# --- Exact 40 features the scaler was trained on (Source excluded) ---
MODEL_FEATURE_COLUMNS = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
    'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length',
    'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Max', 'Idle Mean',
]

# --- Column mapping: 2018 CSV abbreviated name -> model feature name ---
COLUMN_MAP = {
    'Flow Duration':     'Flow Duration',
    'Tot Fwd Pkts':      'Total Fwd Packets',
    'Tot Bwd Pkts':      'Total Backward Packets',
    'TotLen Fwd Pkts':   'Total Length of Fwd Packets',
    'Fwd Pkt Len Max':   'Fwd Packet Length Max',
    'Fwd Pkt Len Min':   'Fwd Packet Length Min',
    'Fwd Pkt Len Mean':  'Fwd Packet Length Mean',
    'Fwd Pkt Len Std':   'Fwd Packet Length Std',
    'Bwd Pkt Len Max':   'Bwd Packet Length Max',
    'Bwd Pkt Len Mean':  'Bwd Packet Length Mean',
    'Bwd Pkt Len Std':   'Bwd Packet Length Std',
    'Flow Byts/s':       'Flow Bytes/s',
    'Flow Pkts/s':       'Flow Packets/s',
    'Flow IAT Std':      'Flow IAT Std',
    'Flow IAT Max':      'Flow IAT Max',
    'Flow IAT Min':      'Flow IAT Min',
    'Fwd IAT Std':       'Fwd IAT Std',
    'Fwd IAT Min':       'Fwd IAT Min',
    'Bwd IAT Tot':       'Bwd IAT Total',
    'Bwd IAT Std':       'Bwd IAT Std',
    'Bwd IAT Max':       'Bwd IAT Max',
    'Bwd IAT Min':       'Bwd IAT Min',
    'Fwd Header Len':    'Fwd Header Length',
    'Bwd Header Len':    'Bwd Header Length',
    'Bwd Pkts/s':        'Bwd Packets/s',
    'Pkt Len Min':       'Min Packet Length',
    'Pkt Len Max':       'Max Packet Length',
    'Pkt Len Mean':      'Packet Length Mean',
    'Pkt Len Std':       'Packet Length Std',
    'FIN Flag Cnt':      'FIN Flag Count',
    'RST Flag Cnt':      'RST Flag Count',
    'PSH Flag Cnt':      'PSH Flag Count',
    'ACK Flag Cnt':      'ACK Flag Count',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd',
    'Fwd Seg Size Min':  'min_seg_size_forward',
    'Active Max':        'Active Max',
    'Idle Mean':         'Idle Mean',
}

# --- Load artifacts ---
model         = joblib.load('../newOutput/ensemble_model.pkl')
scaler        = joblib.load('../newOutput/scaler.pkl')
label_encoder = joblib.load('../newOutput/label_encoder.pkl')

# --- Load CSV ---
df = pd.read_csv('../PortScan.csv')

# Rename abbreviated 2018 columns to model feature names
df = df.rename(columns=COLUMN_MAP)

# Select exactly the 40 features in the order the scaler expects
X = df[MODEL_FEATURE_COLUMNS].copy()
X = X.apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], 0, inplace=True)
X.dropna(inplace=True)

# Scale and predict
X_scaled = scaler.transform(X)
y_pred_enc = model.predict(X_scaled)
y_pred = label_encoder.inverse_transform(y_pred_enc)

# --- Label map: raw CSV labels -> model class names ---
LABEL_MAP = {
    'Benign':                  'Normal Traffic',
    'DoS attacks-GoldenEye':   'DoS',
    'DoS attacks-Slowloris':   'DoS',
    'DoS attacks-Hulk':        'DoS',
    'DoS attacks-SlowHTTPTest':'DoS',
    'Port Scan':               'Port Scan',
    'FTP-BruteForce':          'Brute Force',
    'SSH-Bruteforce':          'Brute Force',
    'DDOS attack-HOIC':        'DDoS',
    'DDOS attack-LOIC-UDP':    'DDoS',
    'DDoS attacks-LOIC-HTTP':  'DDoS',
}

y_true_mapped = pd.Series(df.loc[X.index, 'Label']).map(LABEL_MAP)
unmapped = y_true_mapped.isna().sum()
if unmapped:
    print(f'WARNING: {unmapped} rows have unmapped labels - dropping them')
    mask = y_true_mapped.notna()
    y_true_mapped = y_true_mapped[mask]
    y_pred = y_pred[mask.values]

present_labels = sorted(set(y_true_mapped))

print(classification_report(y_true_mapped, y_pred, labels=present_labels, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true_mapped, y_pred, labels=present_labels),
    index=present_labels,
    columns=present_labels
)
print(cm)

results = df.loc[y_true_mapped.index, ['Label']].copy()
results['Mapped Label'] = y_true_mapped.values
results['Predicted'] = y_pred
results['Correct'] = results['Mapped Label'] == results['Predicted']
print(f'\nOverall Accuracy: {results["Correct"].mean():.4f}')
print('\nLabel encoder classes:', label_encoder.classes_)
print('CSV labels:', pd.Series(df["Label"]).unique())

KeyError: "['Bwd IAT Std', 'RST Flag Count'] not in index"

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Max', 'Active Min', 'Idle Mean', 'Attack Type']
